# Seminar/kodeøkt - SOK3023: Maskinlæring for økonomer

Dette dokumentet inneholder en steg-for-steg veiledning for hvordan man kan bygge et nevralt nettverk ved hjelp av TensorFlow ved hjelp av et datasett kalt `adults.csv`. Vi skal lage en binær klassifiseringsmodell som sier om en person har en lønn under/lik 50K, eller over 50K.

### 1. Last inn bibliotek
Først importerer vi de nødvendige bibliotekene og legger til en funksjon for å laste opp filen i Google Colab.

In [ ]:
# 1. Last inn bibliotek
import tensorflow as tf             # ML bibliotek
from tensorflow import keras        # API for NN
import numpy as np                  # Numpy
import matplotlib.pyplot as plt     # Plotting
import pandas as pd                 # Pandas for dataframes
from sklearn import preprocessing   # SciKit Learn

### 2. Laster inn datasett
Vi bruker pandas til å laste inn `adults.csv`. Vi definerer også kolonnetitlene manuelt her.

In [ ]:
# Laster inn datasettet fra en CSV-fil
filnavn = "adults.csv"

# Tittel til kolonner
header = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
          'marital-status', 'occupation', 'relationship', 'race', 'sex',
          'capital-gain', 'capital-loss', 'hours-per-week',
          'native-country', 'salary']

# Leser inn filen
df = pd.read_csv(filnavn, names=header)

# Viser de første radene i datasettet
df

### 3. Formatet og shape av dataene
Vi sjekker dimensjonene (rader og kolonner) og datatypene.

In [ ]:
print("Shape:", df.shape)
print("Dtypes:\n", df.dtypes)

### 4. Behandling av manglende verdier
Noen verdier er markert som "?". Disse erstatter vi med NaN og fjerner deretter radene som mangler data. Vi fjerner også kolonnen education-num da education dekker samme info .

In [ ]:
# Sjekk etter '?' i datasettet. I dette datasettet er de ofte formatert som ' ?'
print("Antall '?' per kolonne:")
display((df == ' ?').sum())

In [ ]:
# Erstatter "?" med NaN
df = df.replace('?', np.nan)

# Sjekker hvor mange rader som har NaN (valgfritt)
# df[pd.isnull(df).any(axis=1)].shape

# Kvitter oss med radene som inneholder NaN verdier
df.dropna(inplace=True)

# Fjerner education-num
df.drop('education-num', axis=1, inplace=True)

# Sjekker shape etter data-cleaning ("datavask")
print("Ny shape:", df.shape)

### 5. Forberedelse av kategoriske data
Vi definerer hvilke kolonner som er kategoriske og hvilken som er label (`salary`). Vi lager også en hjelpefunksjon for å se unike verdier .

In [ ]:
categorical_columns = ['workclass', 'education', 'marital-status',
                       'occupation', 'relationship',
                       'race', 'sex', 'native-country']
label_column = ['salary']

def show_unique_values(columns):
    for column in columns:
        uniq = df[column].unique().tolist()
        print(column + " has " + str(len(uniq)) + " values: " + str(uniq))

show_unique_values(categorical_columns)
show_unique_values(label_column)

### 6. Prosessere dataene (Label encoding)
Dette står det masse om i kompendium, men vi ønsker å konvertere `salary` (label) til $0$ og $1$. '<=50K' blir 0 og '>50K' blir 1 .

In [ ]:
def convert_to_int(columns):
    for column in columns:
        unique_values = df[column].unique().tolist()
        dic = {}
        for indx, val in enumerate(unique_values):
            dic[val] = indx
        df[column] = df[column].map(dic).astype(int)
        print(column + " ferdig!")

convert_to_int(label_column)
show_unique_values(label_column)

### 7. One hot encoding
Vi bruker "dummies" for å lage binære kolonner for hver kategori for å unngå kunstig rangering av kategoriske variabler .

In [ ]:
# One hot encoding funksjon
def convert_to_onehot(data, columns):
    dummies = pd.get_dummies(data[columns])
    data = data.drop(columns, axis=1)
    data = pd.concat([data, dummies], axis=1)
    return data

# Utfør one-hot encoding
df = convert_to_onehot(df, categorical_columns)

# Sjekk shape og fordeling av salary
print("Shape etter one-hot:", df.shape)
print("Sum salary:", np.sum(df['salary']))

In [ ]:
df.head()

### 8. Normaliser data
Vi normaliserer numeriske kolonner (`age`, `fnlwgt`, `capital-gain`, `capital-loss`, `hours-per-week`) slik at de får gjennomsnitt $0$ og standardavvik $1$ ved hjelp av StandardScaler .

In [ ]:
normalize_columns = ['age', 'fnlwgt', 'capital-gain',
                     'capital-loss', 'hours-per-week']

def show_values(columns):
    for column in columns:
        max_val = df[column].max()
        min_val = df[column].min()
        mean_val = df[column].mean()
        var_val = df[column].var()

        print(column + ' values=['+str(min_val)+','+str(max_val)+'], mean='+str(mean_val)+' var='+str(var_val))

# Vis verdier før normalisering
show_values(normalize_columns)

In [ ]:
def normalize(columns):
    scaler = preprocessing.StandardScaler()
    df[columns] = scaler.fit_transform(df[columns])

# Utfør normalisering
normalize(normalize_columns)
print("\nEtter normalisering:")
df.head()

### 9. Splitte data inn i training, validation og test sett
Vi bruker `train_test_split` fra Scikit Learn for å dele dataene. Vi bruker 20% til test .

In [ ]:
from sklearn.model_selection import train_test_split

x_data = df.drop('salary', axis=1)
y_labels = df['salary']

X_train, X_test, y_train, y_test = train_test_split(x_data, y_labels, test_size=0.2, shuffle=True)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

### 10. Maskinlæringsmodellen (Multi Layer Perceptron)
Vi definerer en sekvensiell modell med input-lag, flatten, to tette lag med 16 nevroner (sigmoid aktivering), og et output-lag .

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(16, activation='sigmoid'),
    tf.keras.layers.Dense(16, activation='sigmoid'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

### 11. Compile og Fit
Vi bruker `adam` optimizer og `binary_crossentropy` som tapsfunksjon. Vi trener over 10 epochs

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Trener modellen, siden ´batch_size´ ikke er satt til noe her er ´batch_size=32´
# ta også prøv litt ulike valg. Hva gjør ´validation_split=0.2´ tror du?
model.fit(X_train, y_train, epochs=10)

# Evaluerer modellen på testsettet
model.evaluate(X_test, y_test, verbose=2)

### 12. Confusion Matrix
Vi genererer en confusion matrix for å se hvor godt modellen treffer på de ulike klassene .

In [ ]:
# 1. Prediker modellen til test
y_pred = model.predict(X_test)

# Konverter sannsynligheter til binære labels (0 or 1)
y_pred_classes = (y_pred > 0.5).astype(int)

# 2. Generer en confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_classes)

print("Confusion Matrix:")
print(cm)

____________

# Forklaring av teknikkene brukt i koden

Her er en dypere forklaring av de viktigste teknikkene og funksjonene vi brukte i koden.

### 1. One-Hot Encoding
Maskinlæringsmodeller forstår ikke tekst (som "Gift", "Ugift", "Skilt"), de forstår kun tall.

* **Problemet:** Hvis vi bare bytter ut ord med tall (f.eks. Rød=1, Blå=2, Grønn=3), vil modellen tro at det finnes en rangering (at Grønn er "mer verdt" enn Rød). Dette skaper "feilaktige sammenhenger".
* **Løsningen:** One-Hot Encoding lager en ny kolonne for *hver* kategori. Hvis en person er "Gift", får den kolonnen verdien 1, mens alle andre sivilstatus-kolonner får 0.
* **I koden:** Vi brukte `pd.get_dummies` for å gjøre dette automatisk for alle kategoriske variabler.

### 2. Normalisering (StandardScaler)
I datasettet ditt har variablene helt forskjellige skalaer. For eksempel er `age` kanskje mellom 18 og 90, mens `capital-gain` kan være titusenvis av dollar.

* **Hvorfor:** Hvis vi ikke normaliserer, vil variabler med store tallverdier (som kapitalgevinst) dominere "læringen" til nettverket, og modellen vil ignorere variabler med små tall (som alder eller timer per uke).
* **Hvordan (`StandardScaler`):** Vi brukte `StandardScaler` fra Scikit-Learn. Denne funksjonen endrer alle tallene slik at hver kolonne får et gjennomsnitt på 0 og et standardavvik på 1.

Matematisk gjøres dette ved formelen:

  $$z = \frac{x - \mu}{\sigma}$$

  Hvor $x$ er verdien, $\mu$ er gjennomsnittet, og $\sigma$ er standardavviket.

### 3. Splitte data (`train_test_split`)
Vi delte dataene inn i to grupper: **X_train** (til å lære) og **X_test** (til å "sjekke" fasiten).

* **Hensikt:** Vi må teste modellen på data den *aldri har sett før* for å vite om den faktisk har lært mønstrene generelt, eller om den bare har pugget treningsdataene utenat (overfitting).
* **Vår split:** Du satte `test_size=0.2`, som betyr at 80% av dataene brukes til trening, og 20% holdes gjemt til slutt-testen.

### 4. Confusion Matrix
Til slutt i koden skrev vi ut en "Confusion Matrix". Dette er en tabell som forteller oss *hvordan* modellen tar feil, ikke bare *at* den tar feil.

Den viser fire ting:
1.  **True Positives:** Modellen sa ">50K", og det var riktig.
2.  **True Negatives:** Modellen sa "<=50K", og det var riktig.
3.  **False Positives:** Modellen trodde personen tjente mye, men de tjente lite.
4.  **False Negatives:** Modellen trodde personen tjente lite, men de tjente faktisk mye.

Dette hjelper deg å vurdere om modellen er "bra, dårlig eller midt i mellom" på en mer nyansert måte enn bare ren nøyaktighet (accuracy).

# Nå kan du kode videre
Kan du lage en random forests, av samme data?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Opprett modellen
rf_model = RandomForestClassifier(
    n_estimators=500,      # hva betyr dette?
    max_depth=20,          # hva betyr dette?
    min_samples_split = 5, # hva betyr dette?
    random_state=42
)

#### Fortsett din egen kode under ####